In [2]:
import time, threading, statistics
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          ContinuousBatchingConfig, GenerationConfig)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DTYPE = torch.float32
MAX_NEW = 48

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"                  # only used by the static comparison
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).eval()
print("device:", model.device, "| attention:", model.config._attn_implementation)

eos = model.generation_config.eos_token_id
EOS_IDS = eos if isinstance(eos, list) else [eos]

# Greedy decoding, same settings for static and continuous so outputs are comparable
GEN_CFG = GenerationConfig(max_new_tokens=MAX_NEW, do_sample=False,
                           eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)

# Paged KV cache: memory is split into blocks ("pages") of page_size tokens
CB_CONFIG = ContinuousBatchingConfig(
    page_size=16,               # tokens per KV block
    num_blocks=256,             # total blocks -> 256 x 16 = 4096 tokens of KV cache for ALL requests together
    max_batch_tokens=256,       # max tokens processed in one forward step (all requests' prefill + decode)
    max_requests_per_batch=16,  # max requests active in one step
)
print("KV capacity (tokens):", CB_CONFIG.page_size * CB_CONFIG.num_blocks)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

device: cpu | attention: sdpa
KV capacity (tokens): 4096


In [3]:
QUESTIONS = [
    "Answer with one word: what colour is the sky on a clear day?",
    "What is 12 times 7? Reply with only the number.",
    "Write three sentences about why the ocean is salty.",
    "List five fruits, comma separated.",
    "Explain in detail how a refrigerator keeps food cold.",
    "Say hello.",
    "Give a short definition of photosynthesis.",
    "Write a short paragraph about the history of the bicycle.",
    "What is the capital of Japan? One word.",
    "Describe how rain forms, step by step.",
    "Translate 'good morning' into French.",
    "Write a haiku about coffee.",
    "Explain what a database index is and why it speeds up queries.",
    "Name the planet closest to the sun.",
    "Give three tips for writing clean Python code.",
    "Summarise the water cycle in two sentences.",
]

def to_ids(q):
    text = tokenizer.apply_chat_template([{"role": "user", "content": q}],
                                         tokenize=False, add_generation_prompt=True)
    return tokenizer.encode(text, add_special_tokens=False)        # a plain list of ints, NO padding

INPUTS = [to_ids(q) for q in QUESTIONS]
print("prompt lengths:", [len(x) for x in INPUTS])

def useful_len(tokens):
    for i, t in enumerate(tokens):
        if t in EOS_IDS:
            return i + 1
    return len(tokens)

prompt lengths: [44, 44, 39, 36, 40, 32, 37, 40, 39, 38, 37, 36, 43, 37, 38, 39]


In [9]:
@torch.inference_mode()
def static_batch(inputs):
    texts = [tokenizer.decode(x) for x in inputs]
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False)
    out = model.generate(**enc, generation_config=GEN_CFG)
    return [row[enc.input_ids.shape[1]:].tolist() for row in out]

_ = static_batch(INPUTS[:2])                                         # warm-up
t0 = time.perf_counter()
static_out = static_batch(INPUTS)
static_s = time.perf_counter() - t0

_ = model.generate_batch(inputs=INPUTS[:2], generation_config=GEN_CFG,
                         continuous_batching_config=CB_CONFIG)           # warm-up (first run is slower)
t0 = time.perf_counter()
cb_res = model.generate_batch(inputs=INPUTS, generation_config=GEN_CFG,
                              continuous_batching_config=CB_CONFIG)
cb_s = time.perf_counter() - t0
cb_out = [cb_res[f"req_{i}"].generated_tokens for i in range(len(INPUTS))]

static_tokens = sum(useful_len(r) for r in static_out)
cb_tokens = sum(useful_len(r) for r in cb_out)
same = [useful_len(a) == useful_len(b) and a[:useful_len(a)] == b[:useful_len(b)]
        for a, b in zip(static_out, cb_out)]
print(f"same answers (static vs continuous): {sum(same)}/{len(same)}")
print(f"STATIC     (1 padded batch of 16): {static_s:6.2f}s  {static_tokens / static_s:7.1f} useful tok/s")
print(f"CONTINUOUS (generate_batch)      : {cb_s:6.2f}s  {cb_tokens / cb_s:7.1f} useful tok/s")
print("\nexample answer:", repr(tokenizer.decode(cb_out[1][:useful_len(cb_out[1])], skip_special_tokens=True)))

same answers (static vs continuous): 6/16
STATIC     (1 padded batch of 16):  35.17s     13.8 useful tok/s
CONTINUOUS (generate_batch)      :  34.28s     13.2 useful tok/s

example answer: '84'


In [10]:
LATE_DELAY_S = 1.0     # when the late short request arrives (seconds after start)

manager = model.init_continuous_batching(generation_config=GEN_CFG, continuous_batching_config=CB_CONFIG)
manager.start()                                    # starts the background thread that runs the model
t0 = time.perf_counter()
arrived, finished = {}, {}

def submit(rid, question, max_new):
    arrived[rid] = time.perf_counter() - t0
    manager.add_request(input_ids=to_ids(question), request_id=rid, max_new_tokens=max_new)

submit("LONG",    "Write a detailed paragraph about the history of computers.", 96)
submit("SHORT-1", "What is 2 + 2? Reply with only the number.", 96)
threading.Timer(LATE_DELAY_S, submit,
                args=("SHORT-LATE", "Name the planet closest to the sun. One word.", 96)).start()

while len(finished) < 3:
    res = manager.get_result(timeout=0.1)          # returns ONE finished request (or None)
    if res is not None and res.is_finished():
        finished[res.request_id] = (time.perf_counter() - t0, useful_len(res.generated_tokens),
                                    tokenizer.decode(res.generated_tokens, skip_special_tokens=True))
manager.stop(block=True)

print(f"{'request':>10} | {'arrived s':>9} | {'finished s':>10} | {'tokens':>6} | answer")
for rid in ["LONG", "SHORT-1", "SHORT-LATE"]:
    done_s, n, text = finished[rid]
    print(f"{rid:>10} | {arrived[rid]:>9.2f} | {done_s:>10.2f} | {n:>6} | {text[:50]!r}")
print("\nfinish order:", sorted(finished, key=lambda r: finished[r][0]))

   request | arrived s | finished s | tokens | answer
      LONG |      0.00 |      20.97 |     96 | 'The history of computers is a fascinating journey '
   SHORT-1 |      0.00 |       1.20 |      2 | '4'
SHORT-LATE |      1.00 |       2.22 |      3 | 'Mercury'

finish order: ['SHORT-1', 'SHORT-LATE', 'LONG']


In [11]:
%%writefile server_cb.py
import asyncio, os, threading, time, uuid
from contextlib import asynccontextmanager

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM, ContinuousBatchingConfig, GenerationConfig

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
DTYPE = getattr(torch, os.environ.get("DTYPE", "float32"))
state = {}


class GenRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 48


def useful_len(tokens, eos_ids):
    for i, t in enumerate(tokens):
        if t in eos_ids:
            return i + 1
    return len(tokens)


def dispatcher(stop_event):
    """Plain thread: takes finished requests from the manager and wakes up the waiting HTTP handler."""
    manager, loop, pending = state["manager"], state["loop"], state["pending"]
    while not stop_event.is_set():
        res = manager.get_result(timeout=0.05)            # one finished request, or None
        if res is None or not res.is_finished():
            continue
        fut = pending.pop(res.request_id, None)
        if fut is None:
            continue
        if res.error:
            loop.call_soon_threadsafe(fut.set_exception, RuntimeError(res.error))
        else:
            loop.call_soon_threadsafe(fut.set_result, res.generated_tokens)


@asynccontextmanager
async def lifespan(app):
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).eval()
    eos = model.generation_config.eos_token_id
    eos_ids = eos if isinstance(eos, list) else [eos]
    gen_cfg = GenerationConfig(max_new_tokens=48, do_sample=False, eos_token_id=eos_ids,
                               pad_token_id=tok.pad_token_id)
    cb_cfg = ContinuousBatchingConfig(page_size=16, num_blocks=int(os.environ.get("NUM_BLOCKS", "256")),
                                      max_batch_tokens=256, max_requests_per_batch=16)
    manager = model.init_continuous_batching(generation_config=gen_cfg, continuous_batching_config=cb_cfg)
    manager.start()                                       # background thread: scheduler + model
    stop_event = threading.Event()
    state.update(tok=tok, eos_ids=set(eos_ids), manager=manager, pending={},
                 loop=asyncio.get_running_loop())
    thread = threading.Thread(target=dispatcher, args=(stop_event,), daemon=True)
    thread.start()
    yield
    stop_event.set()
    manager.stop(block=True)


app = FastAPI(lifespan=lifespan)


@app.get("/health")
async def health():
    return {"status": "ok", "mode": "continuous", "engine_running": state["manager"].is_running()}


@app.post("/generate")
async def generate(req: GenRequest):
    t0 = time.perf_counter()
    tok = state["tok"]
    text = tok.apply_chat_template([{"role": "user", "content": req.prompt}], tokenize=False,
                                   add_generation_prompt=True)
    ids = tok.encode(text, add_special_tokens=False)      # no padding needed
    rid = uuid.uuid4().hex
    fut = state["loop"].create_future()
    state["pending"][rid] = fut
    state["manager"].add_request(input_ids=ids, request_id=rid, max_new_tokens=req.max_new_tokens)
    try:
        tokens = await asyncio.wait_for(fut, timeout=300)
    except asyncio.TimeoutError:
        state["pending"].pop(rid, None)
        state["manager"].cancel_request(rid)
        raise HTTPException(504, "generation timed out")
    n = useful_len(tokens, state["eos_ids"])
    return {"text": tok.decode(tokens[:n], skip_special_tokens=True), "new_tokens": n,
            "server_s": time.perf_counter() - t0}

Writing server_cb.py


In [12]:
import os, sys, json, asyncio, subprocess
import httpx

SERVER_MODEL_ID = MODEL_ID
CONCURRENCY_LEVELS = [1, 4, 8]
N_REQUESTS = 16

def start_server(port):
    env = dict(os.environ, MODEL_ID=SERVER_MODEL_ID, DTYPE=str(DTYPE).replace("torch.", ""))
    log_path = f"server_cb_{port}.log"
    log = open(log_path, "w")
    proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "server_cb:app", "--host", "127.0.0.1",
                             "--port", str(port), "--log-level", "warning"],
                            env=env, stdout=log, stderr=subprocess.STDOUT)
    url = f"http://127.0.0.1:{port}"
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < 600:
        if proc.poll() is not None:
            log.close()
            raise RuntimeError("server crashed:\n" + open(log_path).read()[-3000:])
        try:
            if httpx.get(url + "/health", timeout=2).status_code == 200:
                print(f"server up after {time.perf_counter() - t0:.1f}s:", httpx.get(url + "/health").json())
                return proc, url
        except httpx.HTTPError:
            pass
        time.sleep(1)
    proc.terminate()
    raise TimeoutError("server did not start; see " + log_path)

def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()

async def load_test(url, concurrency, n_requests=N_REQUESTS):
    todo, results = list(range(n_requests)), []
    async def user(client):
        while todo:
            q = QUESTIONS[todo.pop(0) % len(QUESTIONS)]
            t = time.perf_counter()
            r = await client.post(url + "/generate", json={"prompt": q, "max_new_tokens": MAX_NEW})
            r.raise_for_status()
            d = r.json()
            d["latency_s"] = time.perf_counter() - t
            results.append(d)
    async with httpx.AsyncClient(timeout=600) as client:
        r = await client.post(url + "/generate", json={"prompt": "Say hi.", "max_new_tokens": 8})  # warm-up
        r.raise_for_status()
        t0 = time.perf_counter()
        await asyncio.gather(*[user(client) for _ in range(concurrency)])
        wall = time.perf_counter() - t0
    lat = sorted(d["latency_s"] for d in results)
    p = lambda q: lat[min(len(lat) - 1, int(q * len(lat)))]
    return {"concurrency": concurrency, "tok_per_s": sum(d["new_tokens"] for d in results) / wall,
            "req_per_s": len(results) / wall, "p50_s": p(0.5), "p95_s": p(0.95)}

In [13]:
cb_results = []
proc, url = start_server(8010)
try:
    for c in CONCURRENCY_LEVELS:
        r = await load_test(url, c)
        cb_results.append(r)
        print(f"continuous | conc {c:>2} | {r['tok_per_s']:7.1f} tok/s | {r['req_per_s']:5.2f} req/s | "
              f"p50 {r['p50_s']:6.2f}s | p95 {r['p95_s']:6.2f}s")
finally:
    stop_server(proc)

server up after 10.7s: {'status': 'ok', 'mode': 'continuous', 'engine_running': True}
continuous | conc  1 |     4.7 tok/s |  0.17 req/s | p50   9.88s | p95  10.19s
continuous | conc  4 |     9.8 tok/s |  0.35 req/s | p50  14.28s | p95  19.61s
continuous | conc  8 |    12.5 tok/s |  0.44 req/s | p50  18.03s | p95  29.18s


In [15]:
if os.path.exists("results_04_api.json"):
    m4 = json.load(open("results_04_api.json"))
    print(f"{'conc':>4} | {'single tok/s':>12} | {'static-batch tok/s':>18} | {'continuous tok/s':>16} | "
          f"{'static p95':>10} | {'continuous p95':>14}")
    for s, b, c in zip(m4["single"], m4["batch"], cb_results):
        print(f"{c['concurrency']:>4} | {s['tok_per_s']:>12.1f} | {b['tok_per_s']:>18.1f} | {c['tok_per_s']:>16.1f} | "
              f"{b['p95_s']:>9.2f}s | {c['p95_s']:>13.2f}s")
else:
    print("results_04_api.json not found: run this notebook in the same folder as Module 04 to compare.")
    

results_04_api.json not found: run this notebook in the same folder as Module 04 to compare.
